# Local Vector Search with Meta FAISS - Part 1: Indexing & Persistence (Lecture 32)

Meta FAISS (Facebook AI Similarity Search) is the industry standard for high-performance dense vector clustering and similarity search.
This notebook demonstrates:
1. Chunking and generating dense vector representations.
2. Creating an in-memory FAISS index.
3. Persisting the index to local disk (`save_local`) and restoring it (`load_local`).
4. Similarity search with L2 distance / similarity scoring.

## 1. Setup & Ingestion
Initialize embeddings and prepare documents.

In [1]:
import os
import shutil
import tempfile
import numpy as np
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

load_dotenv()
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

documents = [
    Document(page_content="Artificial Intelligence simulates human intelligence through machine algorithms.", metadata={"topic": "ai", "id": 1}),
    Document(page_content="Machine learning enables systems to automatically learn from empirical observations.", metadata={"topic": "ml", "id": 2}),
    Document(page_content="Deep learning architectures utilize multiple layers of abstraction to extract hierarchical representations.", metadata={"topic": "dl", "id": 3}),
    Document(page_content="Natural language processing powers linguistic comprehension, translation, and text generation.", metadata={"topic": "nlp", "id": 4}),
]

splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=15)
chunks = splitter.split_documents(documents)
print(f"Prepared {len(chunks)} chunks.")

C:\Users\aliashraf\AppData\Local\Temp\ipykernel_10988\4172218377.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Prepared 4 chunks.


## 2. FAISS Index Creation & Persistence
Build the FAISS index and serialize it to disk.

In [2]:
temp_faiss_dir = tempfile.mkdtemp(prefix="faiss_test_")

# Build FAISS index
vector_store = FAISS.from_documents(chunks, embeddings)
print(f"FAISS index built with {vector_store.index.ntotal} vectors.")

# Save local
vector_store.save_local(temp_faiss_dir)
print(f"FAISS index saved to: {temp_faiss_dir}")

# Load back with safe deserialization flag
loaded_store = FAISS.load_local(
    temp_faiss_dir,
    embeddings,
    allow_dangerous_deserialization=True
)
print(f"Loaded FAISS index has {loaded_store.index.ntotal} vectors.")
assert loaded_store.index.ntotal == vector_store.index.ntotal, "Vector counts must match" 

FAISS index built with 4 vectors.
FAISS index saved to: C:\Users\ALIASH~1\AppData\Local\Temp\faiss_test_26e9c1x8
Loaded FAISS index has 4 vectors.


## 3. Similarity Search with Score Analysis
Perform queries and evaluate L2 distance metrics.

In [3]:
query = "What is deep learning and how does it use hierarchical layers?"
results_with_scores = loaded_store.similarity_search_with_score(query, k=2)

print(f"Query: {query}\n")
for doc, score in results_with_scores:
    print(f"Score (Distance): {score:.4f} | Document: {doc.page_content} (topic: {doc.metadata.get('topic')})")

best_doc, best_score = results_with_scores[0]
assert best_doc.metadata.get("topic") == "dl", "Top document should match deep learning"
assert best_score >= 0.0, "FAISS L2 distance must be non-negative"

# Cleanup
shutil.rmtree(temp_faiss_dir, ignore_errors=True)
print("FAISS indexing and persistence test passed.")

Query: What is deep learning and how does it use hierarchical layers?

Score (Distance): 0.5747 | Document: Deep learning architectures utilize multiple layers of abstraction to extract hierarchical representations. (topic: dl)
Score (Distance): 1.4392 | Document: Machine learning enables systems to automatically learn from empirical observations. (topic: ml)
FAISS indexing and persistence test passed.
